In [1]:
import pandas as pd
import numpy as np

In [2]:
# Volume-Synchronized Probability of Informed Trading (VPIN) -> lead volatility
def VPIN_volatility(data, window_size=60):
    df = pd.DataFrame()
    df['Total_Bid_Size'] = data[['bid_size_1', 'bid_size_2', 'bid_size_3', 'bid_size_4', 'bid_size_5']].sum(axis=1)
    df['Total_Ask_Size'] = data[['ask_size_1', 'ask_size_2', 'ask_size_3', 'ask_size_4', 'ask_size_5']].sum(axis=1)
    df['OBI'] = (df['Total_Bid_Size'] - df['Total_Ask_Size']) / (df['Total_Bid_Size'] + df['Total_Ask_Size'])
    df['OBI'] = df['OBI'].replace([np.inf, -np.inf], np.nan).fillna(0)
    df['V_B'] = data['volume'] * (1 + df['OBI']) / 2 # estimate V^B
    df['V_S'] = data['volume'] * (1 - df['OBI']) / 2 # estimate V^S
    df['sum_V_B'] = df['V_B'].rolling(window=window_size).sum()
    df['sum_V_S'] = df['V_S'].rolling(window=window_size).sum()
    df['V'] = data['volume'].rolling(window=window_size).mean()
    df['numerator'] = (df['V_B'] - df['V_S']).abs().rolling(window=window_size).sum()
    df['denominator'] = window_size * df['V']
    df['denominator'] = df['denominator'].replace(0, np.nan)
    data['VPIN_volatility'] = df['numerator'] / df['denominator']
    return data